Building a Sentiment Analysis system for movie reviews. Using a dataset of 50,000 IMDb reviews, the project uses a specialized AI called an LSTM (Long Short-Term Memory) network to decide if a review is positive or negative.

Here is the complete explanation of the project in simple language:

1. The Goal
The objective is to train a deep learning model that can read a text review (like "This movie was fantastic!") and automatically classify it as Positive (1) or Negative (0).

2. Why use LSTM?
Standard AI models often look at words individually. However, in a sentence like "The movie was not good," the word "not" completely changes the meaning of "good."

LSTM is a type of "Recurrent Neural Network" (RNN) that has memory.

It understands the sequence of words, meaning it remembers earlier words in a sentence to understand the context of later ones.

3. Project Workflow
Data Collection: The 50,000 IMDb reviews are downloaded from Kaggle using an API(IMDB Dataset of 50K Movie Reviews)(https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews).

Pre-processing: This is the most important step for text.

Encoding: The words "positive" and "negative" are converted to 1s and 0s.

Tokenization: The computer cannot read words, so the top 5,000 most common words are converted into unique numbers (integers).

Padding: Since some reviews are long and some are short, "Padding" is used to make every single review exactly 200 units long so the AI receives a consistent input shape.

Building the Model: The model is built using TensorFlow with three main layers:

Embedding Layer: This places words into a 128-dimensional "map" where words with similar meanings are placed close together.

LSTM Layer: The "brain" of the model that processes the sequence and context of the words.

Dense Layer: The final layer that uses a "Sigmoid" function to give a final probability score (closer to 1 = Positive, closer to 0 = Negative).

4. Training and Evaluation
Training: The model is trained for 5 "epochs" (meaning it reads the entire dataset 5 times).

Validation: 20% of the data is kept aside to test the model during training.

Accuracy: The system achieves about 88-90% accuracy, meaning it correctly guesses the sentiment of a review i.e. 9 out of 10 times.

5. The Predictive System
We build a small function where you can type your own review.

Input: "This movie was fantastic, I loved it!"

Processing: The function tokenizes and pads the text just like the training data.

Output: The model predicts a high probability (e.g., 0.95), and the system prints: "The sentiment is Positive."

6. Important Tips for Developers
GPU Usage: Because LSTMs are complex, the video recommends using a GPU (Graphic Processing Unit) in Google Colab to speed up the training from hours to just a few minutes.

Single Data Point: When testing one review, you must wrap it in a list format so the model knows it is looking at "one data point" rather than individual characters.

In [ ]:
!pip install kaggle

Importing the Dependencies

In [ ]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Data Collection--> Kaggle API

In [ ]:
kaggle_dictionary=json.load(open("kaggle.json"))

In [ ]:
kaggle_dictionary.keys

<function dict.keys>

In [ ]:
#setup kaggle credentials as env variables
os.environ["KAGGLE_USERNAME"]=kaggle_dictionary["username"]
os.environ["KAGGLE_KEY"]=kaggle_dictionary["key"]

In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
  0% 0.00/25.7M [00:00<?, ?B/s]
100% 25.7M/25.7M [00:00<00:00, 870MB/s]


In [ ]:
!ls

imdb-dataset-of-50k-movie-reviews.zip  kaggle.json  sample_data


In [ ]:
# unzip the dataset file
with ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as zip_ref:
  zip_ref.extractall()

In [ ]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


**Loading** **Dataset**

In [ ]:
df = pd.read_csv("/content/IMDB Dataset.csv")

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [ ]:
df.shape

(50000, 2)

In [ ]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [ ]:
df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
1,25000
0,25000


In [ ]:
#split data into trainng and testing data
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
train_data.shape

(40000, 2)

In [ ]:
test_data.shape

(10000, 2)

Data Preprocessing

In [ ]:
# Tokenize text data
tokenizer = Tokenizer(num_words=10000) #This line initializes a Tokenizer object from TensorFlow's Keras API. The num_words=10000 argument tells the tokenizer to consider only the 10,000 most frequently occurring words in your text data. Words beyond this vocabulary size will be ignored.

tokenizer.fit_on_texts(train_data["review"]) #the tokenizer learns the vocabulary from your training reviews (train_data["review"]). It builds an internal word index, mapping each unique word to an integer based on its frequency. This step is only performed on the training data to prevent data leakage from the test set.

X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200) #This line first converts the training reviews into sequences of integers using the learned tokenizer (tokenizer.texts_to_sequences). Each word in a review is replaced by its corresponding integer from the tokenizer's word index. Then, pad_sequences is applied. Since neural networks require input sequences of a fixed length, pad_sequences transforms these integer sequences into uniform length (200 in this case) by adding zeros (padding) to the beginning or end of shorter sequences, and truncating longer ones. The result, X_train, is your preprocessed training input.

X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200) #This line does the same as the previous one, but for your test data (test_data["review"]). It converts test reviews into integer sequences and then pads/truncates them to a fixed length of 200, creating X_test, your preprocessed testing input.

In [ ]:
print(X_train)

[[2946 3749 1828 ...  205  351 3856]
 [   3 4004  208 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  103  125 7285]
 [   0    0    0 ...   70   73 2062]]


In [ ]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [ 133    6  117 ...    7 9050 8494]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [ ]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [ ]:
print(Y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


LSTM - Long Short-Term Memory

In [ ]:
# Building the model

model = Sequential() #This line initializes a Sequential model, which is a linear stack of layers. This is a common and straightforward way to build neural networks in Keras.

model.add(Embedding(input_dim=10000, output_dim=128, input_length=200)) #This adds an Embedding layer.
#(input_dim=5000)-->This is the size of your vocabulary (number of unique words the model will consider). It's typically set to the num_words from your Tokenizer (though in the earlier cell, num_words was 10000, this model uses 5000 for the embedding layer).
#(output_dim=128)--> This defines the dimension of the dense embedding. Each word will be represented by a 128-dimensional vector. Words with similar meanings will have similar embedding vectors.
#(input_length=200)-->This specifies the length of the input sequences (the fixed length after padding), which is 200 in your case. This parameter is deprecated in newer Keras versions but still commonly seen

model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2)) #This adds the core LSTM layer.
#(128)---> This is the number of LSTM units (also known as hidden units or cells) in the layer. It determines the dimensionality of the output space of the LSTM.
#(dropout=0.2)--->This applies dropout to the inputs of the LSTM layer. Dropout is a regularization technique that randomly sets a fraction of input units to zero at each update during training, which helps prevent overfitting.
#(recurrent_dropout=0.2)---> This applies dropout to the recurrent connections (the internal state updates) of the LSTM layer, further aiding in regularization.

model.add(Dense(1, activation="sigmoid")) #This adds a final Dense (fully connected) layer:
#(1)-->This indicates that the layer has a single output unit, as this is a binary classification problem (positive or negative sentiment).
#(activation="sigmoid")-->The sigmoid activation function is used here. It squashes the output to a value between 0 and 1, which can be interpreted as the probability of the sentiment being positive.

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.summary() # is used to print a concise summary of the model's architecture. This output is very useful for understanding the different layers in your neural network, their output shapes, and the total number of trainable parameters in your model. It helps in debugging and understanding the model's complexity.

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# compiling the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])  #This function configures the model for training. You specify the optimizer, loss function, and metrics it should use.
#optimizer='adam'---> The 'adam' optimizer is a popular and efficient algorithm used to update the weights of your neural network during training. It adaptively adjusts learning rates for each parameter, often leading to faster convergence.
#loss='binary_crossentropy'---> This is the loss function used to measure how well the model is performing. For binary classification problems (like sentiment analysis with two classes: positive/negative), binary_crossentropy is the standard choice. It calculates the difference between the predicted probabilities and the true labels.
#metrics=['accuracy']--->This specifies the metrics to monitor during training and testing. 'accuracy' is a common metric that measures the proportion of correctly classified samples. It helps you understand how well your model is predicting the correct sentiment.


Training the Model

In [ ]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 326s 645ms/step - accuracy: 0.7267 - loss: 0.5329 - val_accuracy: 0.8594 - val_loss: 0.3311
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 314s 628ms/step - accuracy: 0.8558 - loss: 0.3432 - val_accuracy: 0.8639 - val_loss: 0.3225
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 317s 634ms/step - accuracy: 0.8783 - loss: 0.3018 - val_accuracy: 0.8634 - val_loss: 0.3436
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 315s 631ms/step - accuracy: 0.9042 - loss: 0.2439 - val_accuracy: 0.8714 - val_loss: 0.3219
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 318s 637ms/step - accuracy: 0.9292 - loss: 0.1878 - val_accuracy: 0.8541 - val_loss: 0.3645


Model Evaluation

In [ ]:
loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 86ms/step - accuracy: 0.8547 - loss: 0.3599
Test Loss: 0.3576451539993286
Test Accuracy: 0.8546000123023987


Building a Predictive System

In [ ]:
def predict_sentiment(review):  #which takes a movie review as input and predicts whether its sentiment is positive or negative using the trained LSTM model.

  # tokenize and pad the review
  sequence = tokenizer.texts_to_sequences([review]) #This line takes the input review (which is a string) and converts it into a sequence of integers. It uses the tokenizer that was fitted on the training data. The [review] wraps the single review in a list, as texts_to_sequences expects a list of texts.

  padded_sequence = pad_sequences(sequence, maxlen=200) #This line then takes the integer sequence and pads or truncates it to a fixed length of 200. This is necessary because the neural network model expects input sequences of a uniform length.

  prediction = model.predict(padded_sequence)  #the preprocessed review (padded_sequence) is fed into the trained model (your LSTM network) to get a prediction. The model outputs a probability score, typically between 0 and 1.

  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"  #this line converts the numerical prediction into a human-readable sentiment. If the model's output probability is greater than 0.5, it's classified as "positive"; otherwise, it's classified as "negative". The [0][0] is used to access the single numerical value from the prediction output.

  return sentiment  #Finally, the function returns the determined sentiment ("positive" or "negative").

In [ ]:
# example usage
new_review = "This movie was fantastic. I loved it."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step
The sentiment of the review is: positive


In [ ]:
# example usage
new_review = "This movie was not that good"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
The sentiment of the review is: positive


In [ ]:
# example usage
new_review = "This movie was ok but not that good."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
The sentiment of the review is: negative
